<!-- CHW_COPILOT_NARRATIVE -->
# 🏥 CHW Copilot — MedGemma Clinical Pipeline

<div style="background: linear-gradient(135deg, #2d6a4f 0%, #40916c 50%, #52b788 100%); padding: 20px; border-radius: 15px; color: white;">
    <h2 style="margin: 0;">🎯 MedGemma Impact Challenge Submission</h2>
    <p style="margin-top: 10px;"><strong>Team:</strong> CHW Copilot | <strong>Model:</strong> MedGemma-4b-it (bfloat16) | <strong>Track:</strong> Agentic Workflow</p>
    <p style="margin-top: 5px;"><strong>Task:</strong> Structured extraction + syndromic surveillance from Community Health Worker notes</p>
</div>

---

## 📋 Table of Contents

1. [Introduction & Problem Statement](#1-introduction)
2. [Why These Two Illnesses](#2-illness-rationale)
3. [CHWs in Syndromic Surveillance](#3-chws-surveillance)
4. [Solution Architecture](#4-solution)
5. [Performance Targets & Limitations](#5-metrics-limitations)
6. [Setup & Dependencies](#6-setup)
7. [Load Prompts & Schemas](#7-prompts)
8. [HuggingFace Authentication](#8-auth)
9. [Load MedGemma](#9-model)
10. [Pipeline Helpers](#10-helpers)
11. [Smoke Test](#11-smoke)
12. [Run Pipeline on Gold Notes](#12-pipeline)
13. [Evaluation](#13-eval)
14. [Surveillance & SITREP](#14-surveillance)
15. [Save Results](#15-save)

---

<!-- CHW_COPILOT_NARRATIVE -->
<a id="1-introduction"></a>
## 1. 🎯 Introduction & Problem Statement

### The Challenge: Community Health Workers and the Surveillance Gap

**Community Health Workers (CHWs)** are the "first-mile" layer of primary care delivery and case-finding, operating at massive scale:

- 🌍 **3.8 million+** CHWs across 98+ countries (Global Fund estimates)
- 🏠 Those same household visits produce the **earliest observable signals** of outbreaks — diarrheal disease clusters, febrile rash, respiratory symptoms
- 📝 But the signal is typically **trapped in informal notes** and lost to delayed aggregation

### The Data Gap

Public health surveillance frameworks emphasize timely detection and response, including both **Indicator-Based Surveillance (IBS)** and **Event-Based Surveillance (EBS)** for early alerting. Yet real-world timeliness remains a persistent bottleneck:

| Problem | Reality |
|---------|---------|
| 📅 **Weekly/monthly reporting** | Many systems depend on weekly or monthly cycles, missing fast-moving outbreaks |
| ⏰ **Timeliness below target** | Studies of weekly epidemic-prone disease reporting show timeliness consistently below target, even with electronic infrastructure |
| 📄 **Paper-first workflows** | Data loss and lag are often worse; paper-to-electronic transition is a core lever for improvement |
| 📊 **Delayed district picture** | District Health Officers see a smoothed, delayed picture — reducing EBS-style early warning value |

### Impact Hypothesis

If a district has **50 CHWs** completing ~10 household encounters/day over ~250 working days, that is **~125,000 encounter notes/year**.

In a weekly reporting regime, even perfect compliance creates an inherent delay (days to a week), and empirical surveillance literature shows delays can extend into **multiple weeks** depending on system level and disease (median reporting delays can reach ~40 days in some contexts).

> **CHW Copilot's goal:** Same-day syndromic aggregation from the raw note (offline), plausibly shifting actionable detection earlier by **5–14 days** in settings that otherwise rely on weekly rollups — while producing **auditable, schema-valid data** rather than opaque model text.

<!-- CHW_COPILOT_NARRATIVE -->
<a id="2-illness-rationale"></a>
## 2. 🔬 Why Respiratory Fever & Acute Watery Diarrhea

### Rationale for Initial Focus

CHW Copilot's initial release deliberately targets **two syndromic categories**: respiratory fever and acute watery diarrhea (AWD). This is not a limitation — it is a strategic design choice driven by three factors:

#### 1. Highest Burden in Sub-Saharan Africa

These are the **two most common syndromic presentations** encountered by CHWs in the regions where this tool is designed to operate. Together, respiratory infections and diarrheal diseases account for the majority of under-5 mortality in low-resource settings.

#### 2. Direct Mapping to IDSR Priority Diseases

Both syndromes map directly to diseases on the **Integrated Disease Surveillance and Response (IDSR)** priority list used across Africa:

| Syndrome Category | IDSR Priority Diseases | Example Pathogens |
|-------------------|----------------------|-------------------|
| 🌡️ **Respiratory Fever** | Measles, Pneumonia, COVID-19, Influenza | *S. pneumoniae*, SARS-CoV-2, Measles virus |
| 💧 **Acute Watery Diarrhea** | Cholera, Typhoid, Dysentery | *V. cholerae*, *S. typhi*, Rotavirus |

#### 3. Clear Case Definitions Enable Reliable Extraction

Both syndromes have well-established, WHO-standard case definitions based on **observable symptoms** (fever + cough + difficulty breathing; watery stool ≥3 times/day). This makes them ideal targets for structured extraction from free-text notes — the symptom vocabulary is consistent and unambiguous.

### Scope to Expand

> **The architecture is deliberately syndrome-agnostic.** Adding a new syndrome category (e.g., acute febrile illness with rash, acute flaccid paralysis, hemorrhagic fever) requires only:
>
> 1. Adding keyword patterns to the deterministic syndrome tagger
> 2. Updating the extraction prompt with relevant symptom fields
> 3. (Optionally) fine-tuning with labeled examples for the new category
>
> No changes to the core pipeline, schema validation, or evidence grounding logic are needed. The system is designed to **grow with surveillance needs**.

<!-- CHW_COPILOT_NARRATIVE -->
<a id="3-chws-surveillance"></a>
## 3. 🏘️ CHWs in Syndromic & Infectious Disease Surveillance

### Why CHWs Are Uniquely Positioned

Community Health Workers are the **only health workforce that operates at the household level** in most low- and middle-income countries. This gives them a unique vantage point:

- 🏠 **Household-level contact** — they observe symptoms, living conditions, and family clusters before anyone presents at a health facility
- 🗣️ **Community trust** — patients share information with CHWs that they might not report to formal healthcare providers
- 🔄 **Regular visits** — routine household visits create a continuous surveillance surface, not just point-in-time facility visits
- 🌍 **Scale** — millions of CHWs conducting millions of visits daily = an enormous, untapped surveillance network

### The Current CHW Journey vs. CHW Copilot

| Step | Today (Paper-first) | With CHW Copilot |
|------|-------------------|-----------------|
| 📝 **Record** | Shorthand / multilingual free-text note | Same — note is the input (typed or voice transcript) |
| 🔄 **Structure** | Manual tallies at end of week | **Instant:** MedGemma extracts structured JSON encounter |
| 🏷️ **Classify** | Manual syndrome assignment (if any) | **Automatic:** keyword + LLM syndrome tagger |
| ✅ **Validate** | None | **Automatic:** JSON Schema Draft-07 + evidence grounding |
| ⚠️ **Alert** | None until district rollup | **Same-day:** red-flag alerts for CHW + anomaly detection |
| 📊 **Aggregate** | Weekly / monthly tallies → district | **Same-day:** structured encounters → weekly counts → anomaly signals |
| 📋 **Report** | District Health Officer manually compiles | **Automated SITREP draft** with MedGemma |

### The Promise of Electronic Surveillance

Pilots of electronic IDSR-style workflows have shown meaningful gains in both **signal capture** and **speed**:
- Higher timeliness and substantially more "rumors" identified/verified from e-reporting sites
- The electronic pathway doesn't just speed up the same process — it **captures signals that paper workflows miss entirely**

CHW Copilot brings this electronic advantage to the **first mile** of the health system, where it has the greatest potential to close the detection gap.

<!-- CHW_COPILOT_NARRATIVE -->
<a id="4-solution"></a>
## 4. 🏗️ Solution Architecture

### Why MedGemma?

This submission is built for the **MedGemma Impact Challenge**, where using at least one HAI-DEF model is mandatory. We leverage **MedGemma 1.5** as the system's core "medical-language engine" because:

| Feature | Description |
|---------|-------------|
| 🏥 **Medical priors** | HAI-DEF models are designed for healthcare, with documented pathways for prompt-based adaptation |
| 📦 **Compute-efficient** | Available in a 4B-class model, suitable for edge deployment |
| 🔓 **Open-weight** | Distributed via Hugging Face — enables offline, privacy-preserving inference |
| 🔧 **Adaptable** | Supports QLoRA fine-tuning for domain-specific improvements |

### The 6-Agent Pipeline

**MedGemma** handles the parts that require medical priors and flexible inference. **Deterministic agents** handle the parts that must be exact and inspectable.

```
CHW note (typed / voice transcript)
  ↓
[1] Encounter Extractor (MedGemma 4B; schema-guided JSON)
  ↓
[2] Evidence Grounder (deterministic: span-match quotes per field)
  ↓
[3] Hallucination Detector (Pythea/Strawberry: evidence-scrub + KL budget gap)
      ↳ if flagged → 1 targeted retry of [1] with "unsupported claims" feedback
  ↓
[4] Syndrome Tagger (MedGemma; deterministic fallback on low confidence)
  ↓
[5] Checklist Generator (MedGemma; missing-field + risk-trigger prompts)
  ↓
[6] Schema Validator (deterministic: JSON Schema Draft-07)
  ↓
Outputs:
  • Encounter JSON (schema-valid + evidence-quoted)
  • Red-flag alerts for CHW
  • Aggregates (weekly counts by syndrome × location; anomaly signals; SITREP draft)
```

### Key Innovations

1. **Verification as a first-class step** — not an afterthought; integrated hallucination detection via Pythea/Strawberry
2. **Self-correction loop** — when the detector flags insufficient evidence, the extractor re-runs with targeted feedback, prioritizing abstention over fabrication
3. **Evidence anchoring** — every extracted field is tied to a verbatim quote from the original note
4. **Schema-enforced output** — invalid outputs cannot be emitted; 100% schema validation pass rate
5. **QLoRA fine-tuning** — 4-bit quantized LoRA adapters reduce schema failures and improve evidence quoting
6. **DHIS2-aligned aggregation** — structured encounters → aggregate indicators, designed for the world's largest HMIS platform (~80 LMICs, ~3.2B people)

<!-- CHW_COPILOT_NARRATIVE -->
<a id="5-metrics-limitations"></a>
## 5. 📊 Performance Targets & Limitations

### Performance Metrics

| Metric | Target | How Measured / Enforced |
|--------|--------|----------------------|
| 🏷️ **Syndrome tag accuracy** | ≥80% | Labeled eval set; reported in this notebook |
| 📎 **Evidence grounding rate** | ≥98% | Deterministic quote requirement; unsupported fields coerced to `unknown` |
| ⚠️ **Hallucination rate** | ≤2% | Strawberry "budget gap" flags + abstention |
| ✅ **Schema validation pass rate** | 100% | Strict Draft-07 validation gate |
| ⏱️ **Avg processing time / note** | <10s (GPU) | Measured on T4-class GPU |

### Deployment Plan (Offline-first, Privacy-focused)

| Phase | Description |
|-------|-------------|
| 🔬 **Prototype** | Single-machine demo on T4-class GPU, no network dependencies beyond initial model download |
| 🏭 **Production** | Edge workstation or district laptop with quantized model + store-and-forward sync |
| 🔒 **Privacy** | Local inference by default; only structured aggregates synced; minimum cell size of 5 for published aggregates |

### Limitations (Explicit & Testable)

> **CHW Copilot is NOT a diagnosis engine.** It produces surveillance-grade structure from notes and always preserves uncertainty where evidence is missing.

**Known weaknesses:**
1. 🌐 **Multilingual shorthand** — extremely noisy multilingual free-text without reliable transcription is the weakest mode
2. ⏰ **Temporal reasoning** — merging multiple visits in a single note can confuse time anchoring
3. 🌍 **Population-specific** — model adaptation is workflow- and population-specific; HAI-DEF guidance emphasizes use-case validation before deployment

**Mitigations:**
- Checklist-driven clarification prompts the CHW for missing information
- Conservative abstention + re-run logic rather than forced completion
- QLoRA fine-tuning with region-specific medical QA (AfriMed-QA)

---

<a id="6-setup"></a>
## 🔧 0. Setup

In [11]:
import os, json, sys, time, warnings
from pathlib import Path
import pandas as pd
import torch
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

IS_KAGGLE = os.path.exists("/kaggle/working")

if IS_KAGGLE:
    ROOT = list(Path("/kaggle/input/datasets").iterdir())[0]
else:
    ROOT = Path("..")

sys.path.insert(0, str(ROOT))
OUT_DIR = Path("/kaggle/working") if IS_KAGGLE else Path(".")

print(f"Root:   {ROOT}")
print(f"Output: {OUT_DIR}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)} ({torch.cuda.mem_get_info()[0]/1e9:.1f} GB free)")
else:
    print("⚠️  No GPU — enable T4 in Notebook → Accelerator")


RuntimeError: ❌ Could not find project files. Attach the dataset.

In [13]:
!find /kaggle/input -maxdepth 6 -type d


/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/kheziantomoa


In [9]:
import os, json, sys, time, warnings
from pathlib import Path
import pandas as pd
import torch
warnings.filterwarnings("ignore")

IS_KAGGLE = os.path.exists("/kaggle/working")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

ROOT = None
if IS_KAGGLE:
    # Debug: show what's in /kaggle/input/
    print("📂 Contents of /kaggle/input/:")
    for p in sorted(Path("/kaggle/input").iterdir()):
        print(f"   {p.name}/ ({len(list(p.iterdir()))} items)" if p.is_dir() else f"   {p.name}")
    
    # Try git clone first
    cloned = Path("/kaggle/working/chw-copilot")
    if cloned.exists():
        ROOT = cloned
        print("✅ Using git-cloned repo")
    else:
        # Search all input datasets for our project structure
        for candidate in sorted(Path("/kaggle/input").rglob("prompts")):
            if candidate.is_dir() and (candidate.parent / "src").exists():
                ROOT = candidate.parent
                print(f"✅ Dataset found at: {ROOT}")
                break
        # Fallback: check top-level dataset dirs directly
        if ROOT is None:
            for d in sorted(Path("/kaggle/input").iterdir()):
                if d.is_dir() and (d / "prompts").exists():
                    ROOT = d
                    print(f"✅ Dataset found (top-level): {ROOT}")
                    break
    
    if ROOT is None:
        raise RuntimeError("❌ Could not find project files. Attach the chw-copilot dataset.")
else:
    ROOT = Path("..")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUT_DIR = Path("/kaggle/working") if IS_KAGGLE else Path(".")

print(f"Root:   {ROOT}")
print(f"Output: {OUT_DIR}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)} ({torch.cuda.mem_get_info()[0]/1e9:.1f} GB free)")
else:
    print("⚠️  No GPU detected — enable T4 in Notebook → Accelerator")


📂 Contents of /kaggle/input/:
   datasets/ (1 items)


RuntimeError: ❌ Could not find project files. Attach the chw-copilot dataset.

In [ ]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.40", "accelerate>=0.27", "jsonschema>=4.17"])
print("Dependencies installed ✅")


## 📂 1. Load Prompts & Schemas

In [ ]:
import jsonschema

def load_prompt(name: str) -> str:
    path = ROOT / "prompts" / f"{name}.txt"
    if not path.exists():
        raise FileNotFoundError(f"Prompt not found: {path}")
    return path.read_text(encoding="utf-8")

def load_schema(name: str) -> dict:
    path = ROOT / "schemas" / f"{name}.schema.json"
    with open(path, encoding="utf-8") as f:
        return json.load(f)

# Load prompts
combined_prompt  = load_prompt("combined_pipeline")   # extraction + checklist in one call
sitrep_prompt    = load_prompt("monitoring_sitrep")

# Load validation schemas
encounter_schema = load_schema("encounter")
syndrome_schema  = load_schema("syndrome")
checklist_schema = load_schema("checklist")
sitrep_schema    = load_schema("sitrep")

print(f"✅ Prompts loaded  — combined: {len(combined_prompt)} chars, sitrep: {len(sitrep_prompt)} chars")
print(f"✅ Schemas loaded  — encounter, syndrome, checklist, sitrep")


## 🔑 2. HuggingFace Authentication

> MedGemma is a **gated model**. Accept the licence at [huggingface.co/google/medgemma-4b-it](https://huggingface.co/google/medgemma-4b-it) then add your token via Add-ons → Secrets → `HF_TOKEN`.

In [ ]:
HF_TOKEN = None
if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    try:
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded ✅")
    except Exception:
        print("⚠️  HF_TOKEN not found — MedGemma may fail to load")
else:
    HF_TOKEN = os.getenv("HF_TOKEN")
    print("HF_TOKEN loaded from env ✅" if HF_TOKEN else "⚠️  No HF_TOKEN set")


## 🤖 3. Load MedGemma-4b-it

The model is loaded in **bfloat16 precision** (~8 GB VRAM on T4). No quantisation libraries required.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

MEDGEMMA_ID = "google/medgemma-4b-it"
print(f"Loading {MEDGEMMA_ID} (bfloat16)...")
t0 = time.time()

mg_tokenizer = AutoProcessor.from_pretrained(MEDGEMMA_ID, token=HF_TOKEN)

mg_model = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN,
)

mg_model.eval()
device = next(mg_model.parameters()).device
print(f"\u2705 MedGemma ready  \u2014  device: {device}  |  dtype: bfloat16  |  loaded in {time.time()-t0:.1f}s")


## 🛠️ 4. Pipeline Helpers

All processing logic lives in `src/pipeline_helpers.py` — keeping this notebook clean and readable.

In [ ]:
from src.pipeline_helpers import (
    parse_json_response,
    run_medgemma,
    process_note,
    process_notes_batch,
    keyword_syndrome_tag,
)

# Bind model + tokenizer into a convenient wrapper
def generate(prompt, max_new_tokens=512):
    return run_medgemma(prompt, mg_model, mg_tokenizer, max_new_tokens)

print("✅ Helpers imported from src/pipeline_helpers")


## 🔬 5. Smoke Test — Single Note

Quick sanity check before running the full pipeline.

In [7]:
TEST_NOTE = "Child 3yo male fever 3 days cough bad difficulty breathing rash on chest no diarrhea mother says not eating gave ORS referred health center"

print("Processing test note...")
t0 = time.time()
test_result = process_note(
    note_text    = TEST_NOTE,
    encounter_id = "smoke_test",
    location_id  = "loc_test",
    week_id      = 0,
    combined_prompt = combined_prompt,
    model        = mg_model,
    tokenizer    = mg_tokenizer,
)
elapsed = time.time() - t0

enc = test_result["encounter"]
syn = test_result["syndrome_tag"]

print(f"\n{'─'*50}")
print(f"  Syndrome tag  : {syn['syndrome_tag']}  ({syn['confidence']} confidence)")
print(f"  Sub-syndrome  : {syn.get('sub_syndrome', '—')}")
print(f"  Triggers      : {', '.join(syn['trigger_quotes'])}")
print(f"  Fever         : {enc['symptoms']['fever']['value']}")
print(f"  Cough         : {enc['symptoms']['cough']['value']}")
print(f"  Diff. breath  : {enc['symptoms']['difficulty_breathing']['value']}")
print(f"  Patient       : {enc['patient'].get('age_years', '?')}yo {enc['patient'].get('sex', '?')}")
print(f"  Severity      : {enc['severity']}")
print(f"  Onset (days)  : {enc.get('onset_days', '?')}")
print(f"  Onset week    : {enc.get('estimated_onset_week', '?')}")
print(f"  Errors        : {test_result.get('errors', []) or 'none'}")
print(f"{'─'*50}")
print(f"  ⏱  {elapsed:.1f}s")
print(f"\n  📋 Recommendations:")
for rec in test_result.get('recommendations', []):
    print(f"    {rec}")


Processing test note...


NameError: name 'process_note' is not defined

## ⚙️ 6. Run Pipeline on Gold Notes

In [ ]:
# Load gold notes
gold_path = ROOT / "data_synth" / "gold_encounters_merged.jsonl"
if not gold_path.exists():
    gold_path = ROOT / "data_synth" / "gold_encounters.jsonl"

gold_notes = [json.loads(l) for l in open(gold_path, encoding="utf-8")]

N_DEMO     = len(gold_notes)   # full run on all gold notes
BATCH_SIZE = 4    # notes processed in parallel; reduce to 2 if OOM

# Stratified random sample — ensures all syndrome types are represented
import random, collections
random.seed(42)
by_syndrome = collections.defaultdict(list)
for n in gold_notes:
    by_syndrome[n.get("gold_syndrome_tag", "unclear")].append(n)

demo_notes = []
syndromes  = list(by_syndrome.keys())
# Round-robin fill up to N_DEMO in syndrome order, then shuffle
per_syndrome = max(1, N_DEMO // len(syndromes))
for syn, pool in by_syndrome.items():
    demo_notes.extend(random.sample(pool, min(per_syndrome, len(pool))))
# Top up if needed
remaining = [n for n in gold_notes if n not in demo_notes]
random.shuffle(remaining)
demo_notes.extend(remaining[:max(0, N_DEMO - len(demo_notes))])
random.shuffle(demo_notes)
demo_notes = demo_notes[:N_DEMO]

tag_counts = collections.Counter(n.get("gold_syndrome_tag","unclear") for n in demo_notes)
print(f"Loaded {len(gold_notes)} gold notes | Stratified sample {N_DEMO}: {dict(tag_counts)}")


In [ ]:
t0 = time.time()
results = process_notes_batch(
    notes           = demo_notes,
    combined_prompt = combined_prompt,
    model           = mg_model,
    tokenizer       = mg_tokenizer,
    batch_size      = BATCH_SIZE,
)
avg_t = (time.time() - t0) / len(results)
print(f"\n✅ Done — {len(results)} notes  |  avg {avg_t:.1f}s/note  |  total {time.time()-t0:.0f}s")


## 📊 7. Evaluation

In [ ]:
# Syndrome accuracy
correct = sum(1 for r, g in zip(results, demo_notes)
              if r['syndrome_tag']['syndrome_tag'] == g.get('gold_syndrome_tag','unclear'))
accuracy = correct / len(results)

# Evidence grounding
total_claims = grounded = 0
for r in results:
    note_lower = r['encounter']['note_text'].lower()
    for v in r['encounter']['symptoms'].values():
        if v.get('value') == 'yes':
            total_claims += 1
            q = v.get('evidence_quote','')
            if q and q.lower() in note_lower:
                grounded += 1

grounding_rate = grounded / total_claims if total_claims else 0.0

# Summary table
summary = pd.DataFrame([{
    "Metric": "Syndrome Tag Accuracy",
    "Value":  f"{accuracy:.1%}  ({correct}/{len(results)})",
}, {
    "Metric": "Evidence Grounding Rate",
    "Value":  f"{grounding_rate:.1%}  ({grounded}/{total_claims} claims)",
}, {
    "Metric": "Avg Processing Time",
    "Value":  f"{avg_t:.1f}s / note",
}, {
    "Metric": "Notes Processed",
    "Value":  str(len(results)),
}])

print(summary.to_string(index=False))


In [ ]:
# Confusion matrix
from collections import defaultdict
confusion = defaultdict(int)
for r, g in zip(results, demo_notes):
    confusion[(g.get('gold_syndrome_tag','unclear'), r['syndrome_tag']['syndrome_tag'])] += 1

actuals = sorted(set(k[0] for k in confusion))
preds   = sorted(set(k[1] for k in confusion))

header = f"{'':>28}" + "".join(f"{p:>22}" for p in preds)
print(header)
for a in actuals:
    row = f"{a:>28}" + "".join(f"{confusion.get((a,p),0):>22}" for p in preds)
    print(row)


### 📋 Sample Predictions

In [ ]:
# Show a sample of notes with their predicted labels — good for spot-checking
N_SAMPLE = min(5, len(results))
rows = []
for r, g in zip(results[:N_SAMPLE], demo_notes[:N_SAMPLE]):
    enc = r["encounter"]
    syn = r["syndrome_tag"]
    rows.append({
        "ID":          r["encounter_id"],
        "Note (first 80 chars)": g["note_text"][:80] + "…",
        "Predicted":   syn["syndrome_tag"],
        "Sub-syndrome": syn.get("sub_syndrome", "—"),
        "Confidence":  syn["confidence"],
        "Gold label":  g.get("gold_syndrome_tag", "—"),
        "Match":       "✅" if syn["syndrome_tag"] == g.get("gold_syndrome_tag") else "❌",
        "Fever":       enc["symptoms"]["fever"]["value"],
        "Cough":       enc["symptoms"]["cough"]["value"],
        "Diff.breath": enc["symptoms"]["difficulty_breathing"]["value"],
        "Severity":    enc["severity"],
    })

sample_df = pd.DataFrame(rows)
sample_df


### 🩺 Recommendations & Follow-Up Questions

The pipeline generates WHO ICCM-aligned recommendations and identifies missing clinical information that the CHW should follow up on.

In [ ]:
# Show recommendations and follow-up questions for sample cases
print("=" * 60)
print("📋  SAMPLE RECOMMENDATIONS & FOLLOW-UP QUESTIONS")
print("=" * 60)

N_REC_SAMPLE = min(5, len(results))
for r, g in zip(results[:N_REC_SAMPLE], demo_notes[:N_REC_SAMPLE]):
    enc = r["encounter"]
    syn = r["syndrome_tag"]
    recs = r.get("recommendations", [])
    
    print(f"\n{'─'*60}")
    print(f"  📌 {r['encounter_id']}  |  {syn['syndrome_tag']}  ({syn['confidence']})")
    print(f"  Note: {g['note_text'][:100]}…")
    print(f"  Severity: {enc['severity']}  |  Red flags: {', '.join(str(rf) for rf in enc.get('red_flags', [])) or 'none'}")
    print(f"\n  🩺 Recommendations:")
    for rec in recs:
        print(f"    {rec}")
    
    # Show what information is missing (follow-up checklist)
    missing = []
    if enc["patient"].get("age_years") is None and enc["patient"].get("age_months") is None:
        missing.append("How old is the patient?")
    if enc["patient"].get("sex") in (None, "unknown"):
        missing.append("Is the patient male or female?")
    if enc.get("onset_days") is None:
        missing.append("When did the symptoms start?")
    if enc["symptoms"].get("fever", {}).get("value") == "unknown":
        missing.append("Does the patient have fever or hot body?")
    if enc["symptoms"].get("watery_diarrhea", {}).get("value") == "unknown":
        missing.append("Does the patient have watery diarrhoea?")
    if enc["symptoms"].get("difficulty_breathing", {}).get("value") == "unknown":
        missing.append("Does the patient have difficulty breathing or fast breathing?")
    if enc.get("referral") is None and not any("REFER" in r for r in recs):
        missing.append("Should this patient be referred to the health facility?")
    
    if missing:
        print(f"\n  ❓ Follow-up questions for CHW:")
        for m in missing:
            print(f"    • {m}")
    else:
        print(f"\n  ✅ All key clinical fields captured")

print(f"\n{'─'*60}")
print(f"\n💡 These recommendations follow WHO ICCM guidelines for community-level case management.")


### 📈 Visualisations

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

plt.style.use('dark_background')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Chart 1: Syndrome distribution ────────────────────────────────────────────
tags = [r['syndrome_tag']['syndrome_tag'] for r in results]
tag_counts = Counter(tags)
colors = {
    'respiratory_fever': '#e74c3c',
    'acute_watery_diarrhea': '#3498db',
    'other': '#95a5a6',
    'unclear': '#7f8c8d',
}
bars = axes[0].bar(
    tag_counts.keys(),
    tag_counts.values(),
    color=[colors.get(t, '#95a5a6') for t in tag_counts.keys()],
    edgecolor='white', linewidth=0.5
)
axes[0].set_title('Syndrome Distribution', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Count')
for bar, count in zip(bars, tag_counts.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(count), ha='center', fontweight='bold', fontsize=11)

# ── Chart 2: Symptom extraction quality ───────────────────────────────────────
sym_names = ['fever', 'cough', 'watery_diarrhea', 'difficulty_breathing', 'vomiting', 'rash']
yes_counts = []
unknown_counts = []
no_counts = []
for sym in sym_names:
    vals = [r['encounter']['symptoms'][sym]['value'] for r in results]
    yes_counts.append(vals.count('yes'))
    unknown_counts.append(vals.count('unknown'))
    no_counts.append(vals.count('no'))

x = range(len(sym_names))
w = 0.25
axes[1].bar([i - w for i in x], yes_counts, w, label='yes', color='#2ecc71', edgecolor='white', linewidth=0.5)
axes[1].bar(x, unknown_counts, w, label='unknown', color='#f39c12', edgecolor='white', linewidth=0.5)
axes[1].bar([i + w for i in x], no_counts, w, label='no', color='#e74c3c', edgecolor='white', linewidth=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels([s.replace('_', '\n') for s in sym_names], fontsize=9)
axes[1].set_title('Symptom Extraction Quality', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Count')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'pipeline_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("Charts saved ✅")


In [ ]:
# Weekly surveillance trends (from sim_events)
try:
    sim_path_viz = ROOT / "data_synth" / "sim_events.csv"
    if sim_path_viz.exists():
        sim_viz = pd.read_csv(sim_path_viz)
        syn_col = next((c for c in sim_viz.columns if 'syndrome' in c.lower()), None)

        fig, ax = plt.subplots(figsize=(12, 5))
        for syndrome, grp in sim_viz.groupby(syn_col):
            weekly = grp.groupby('week_id').size()
            color = '#e74c3c' if 'resp' in str(syndrome).lower() else '#3498db' if 'diarr' in str(syndrome).lower() else '#95a5a6'
            ax.plot(weekly.index, weekly.values, 'o-', label=syndrome, color=color, linewidth=2, markersize=4)

        ax.set_xlabel('Week', fontsize=12)
        ax.set_ylabel('Case Count', fontsize=12)
        ax.set_title('Weekly Syndrome Surveillance Trends', fontweight='bold', fontsize=14)
        ax.legend(fontsize=10)
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUT_DIR / 'surveillance_trends.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("Surveillance chart saved ✅")
    else:
        print("⚠️  sim_events.csv not found — skipping surveillance chart")
except Exception as e:
    print(f"⚠️  Chart error: {e}")


## 🚨 8. Surveillance — Anomaly Detection & SITREP

In [ ]:
from src.detect import detect_anomalies

# Map location IDs to real names
LOCATION_NAMES = {
    "loc01": "Gicumbi Health Centre",
    "loc02": "Gakenke Health Centre",
    "loc03": "Nyanza Health Centre",
    "loc04": "Ngoma Health Centre",
    "loc05": "Kibera Health Post",
    "loc06": "Mathare Health Post",
    "loc07": "Dandora Health Post",
    "loc08": "Korogocho Health Post",
}

sim_path = ROOT / "data_synth" / "sim_events.csv"
if not sim_path.exists():
    print("⚠️  sim_events.csv not found — skipping surveillance demo")
else:
    sim_df = pd.read_csv(sim_path)
    sim_df["location_name"] = sim_df["location_id"].map(LOCATION_NAMES).fillna(sim_df["location_id"])
    syndrome_col = next((c for c in sim_df.columns if "syndrome" in c.lower()), None)

    weekly_counts = (
        sim_df
        .groupby(["location_id", syndrome_col, "week_id"], dropna=False)
        .size().reset_index(name="count")
        .rename(columns={syndrome_col: "syndrome_tag"})
    )
    weekly_counts["week_id"] = pd.to_numeric(weekly_counts["week_id"]).astype(int)

    anomalies = detect_anomalies(weekly_counts)
    print(f"Simulation events: {len(sim_df):,}  |  Weekly count rows: {len(weekly_counts)}")
    print(f"\n🚨 Anomalies detected: {len(anomalies)}")
    if not anomalies.empty:
        print(anomalies.to_string(index=False))


In [ ]:
try:
    # Generate SITREP for the peak outbreak week
    if sim_path.exists() and not anomalies.empty:
        outbreak_week  = anomalies["week_id"].max()
        week_anomalies = anomalies[anomalies["week_id"] == outbreak_week]
        week_counts    = sim_df[sim_df["week_id"] == outbreak_week]

        sitrep_p = (sitrep_prompt
                    .replace("{anomalies_csv}",     week_anomalies.to_csv(index=False))
                    .replace("{weekly_counts_csv}", week_counts.to_csv(index=False))
                    .replace("{report_week}",       str(outbreak_week)))

        print(f"Generating SITREP for week {outbreak_week}...")
        t0 = time.time()
        raw_sitrep = generate(sitrep_p, max_new_tokens=2048)
        sitrep     = parse_json_response(raw_sitrep)

        if sitrep:
            print(f"\n{'─'*50}")
            print(f"  Week        : {sitrep.get('report_week')}")
            print(f"  Alerts      : {len(sitrep.get('alerts', []))}")
            print(f"  Summary     : {sitrep.get('summary_text','')[:200]}")
            print(f"{'─'*50}")
            print(f"  ⏱  {time.time()-t0:.1f}s")
            # Save SITREP
            sitrep_path = OUT_DIR / f"sitrep_week{outbreak_week}.json"
            sitrep_path.write_text(json.dumps(sitrep, indent=2), encoding="utf-8")
            print(f"  Saved → {sitrep_path}")
        else:
            print("⚠️  SITREP parse failed. Raw output:")
            print(raw_sitrep[:500])
except Exception as e:
    print(f"⚠️  SITREP section failed: {e} — results already saved above")


## 💾 9. Save Results

In [ ]:
output = {
    "model":                  MEDGEMMA_ID,
    "precision":              "bfloat16",
    "n_notes_processed":      len(results),
    "avg_processing_time_s":  round(avg_t, 2),
    "syndrome_accuracy":      round(accuracy, 3),
    "evidence_grounding_rate":round(grounding_rate, 3),
    "encounters":             [r["encounter"]       for r in results],
    "syndrome_tags":          [r["syndrome_tag"]     for r in results],
    "recommendations":        [r["recommendations"]  for r in results],
}

out_path = OUT_DIR / "pipeline_results.json"
out_path.write_text(json.dumps(output, indent=2, default=str), encoding="utf-8")

print("=" * 52)
print("🎉  CHW Copilot pipeline complete!")
print("=" * 52)
print(f"  Notes processed   : {len(results)}")
print(f"  Syndrome accuracy : {accuracy:.1%}")
print(f"  Evidence grounding: {grounding_rate:.1%}")
print(f"  Avg time / note   : {avg_t:.1f}s")
print(f"  Quantisation      : bfloat16")
print(f"  Results saved to  : {out_path}")
